# Titanic: data preparation without leakage

**Goal:** measure honestly what each preparation step adds. This is a corrected version of the first notebook (kept as `titanic_data_preparation_original.ipynb`). What changed:

1. every step that *learns* from data (imputer, scaler, clipping fences, encoder, feature selector, SMOTE) lives **inside a Pipeline**, so it is re-fitted on the training part of every cross-validation fold;
2. repeated stratified 5-fold CV (4 repeats) with mean and standard deviation, instead of one 5-fold run;
3. one-hot encoding by `OneHotEncoder(handle_unknown="ignore")` instead of `get_dummies` before the split, and no `LabelEncoder` on a feature;
4. the 'title' feature of the first version was *not* a title (the seaborn data has no name; it was a copy of sex and age), so it is removed;
5. the results table is computed, not claimed: the first version said feature engineering gives +3 to +6 %; measured here it gives about 0.

In [1]:
import warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel, SelectKBest, f_classif
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_val_score, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, RobustScaler, StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

try:
    T = pd.read_csv("titanic.csv")                      # shipped next to this notebook
except FileNotFoundError:
    import seaborn as sns; T = sns.load_dataset("titanic")
y = T["survived"].to_numpy()
print(T.shape, "| survived:", round(y.mean(), 3))

(891, 15) | survived: 0.384


## 1. Inspect before you clean

In [2]:
print(T.dtypes.to_string())
print((T.isna().mean() * 100).round(1)[lambda s: s > 0].to_string())
T[["age", "fare", "sibsp", "parch"]].describe().round(2)

survived         int64
pclass           int64
sex                str
age            float64
sibsp            int64
parch            int64
fare           float64
embarked           str
class              str
who                str
adult_male        bool
deck               str
embark_town        str
alive              str
alone             bool
age            19.9
embarked        0.2
deck           77.2
embark_town     0.2


,age,fare,sibsp,parch
count,714.00,891.00,891.00,891.00
mean,29.70,32.20,0.52,0.38
std,14.53,49.69,1.10,0.81
min,0.42,0.00,0.00,0.00
25%,20.12,7.91,0.00,0.00
50%,28.00,14.45,0.00,0.00
75%,38.00,31.00,1.00,0.00
max,80.00,512.33,8.00,6.00


`deck` is 77 % missing and is dropped; `age` (20 %) is imputed; the two missing `embarked` values get the mode. `class`, `who`, `embark_town`, `alive`, `adult_male`, `alone` repeat other columns (`alive` *is* the target), so we model only `pclass, sex, age, sibsp, parch, fare, embarked`.

## 2. Building blocks (all fitted on the training fold only)

In [3]:
def engineer(X):
    # row-wise features: nothing is learned from the data, so this step cannot leak
    X = X.copy()
    X["family_size"] = X["sibsp"] + X["parch"] + 1
    X["is_alone"] = (X["family_size"] == 1).astype(int)
    X["fare_per_person"] = X["fare"] / X["family_size"]
    X["age_missing"] = X["age"].isna().astype(int)
    return X

class IQRClipper(BaseEstimator, TransformerMixin):
    # winsorise to Q1 - k*IQR .. Q3 + k*IQR; the fences come from the training fold only
    def __init__(self, k=1.5): self.k = k
    def fit(self, X, y=None):
        q1, q3 = np.nanpercentile(np.asarray(X, float), [25, 75], axis=0)
        self.lo_, self.hi_ = q1 - self.k * (q3 - q1), q3 + self.k * (q3 - q1); return self
    def transform(self, X): return np.clip(np.asarray(X, float), self.lo_, self.hi_)

def make(stage, model="rf"):
    num, steps = ["pclass", "age", "sibsp", "parch", "fare"], []
    if stage >= 5:
        steps.append(("fe", FunctionTransformer(engineer)))
        num = num + ["family_size", "is_alone", "fare_per_person", "age_missing"]
    ns = [("imp", SimpleImputer(strategy="median") if stage == 0 else KNNImputer(n_neighbors=5))]
    if stage >= 4: ns.append(("clip", IQRClipper()))
    if stage >= 3: ns.append(("scale", StandardScaler() if model == "lr" else RobustScaler()))
    tf = [("num", Pipeline(ns), num)]
    if stage >= 2:
        tf.append(("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                    ("oh", OneHotEncoder(handle_unknown="ignore"))]), ["sex", "embarked"]))
    steps.append(("ct", ColumnTransformer(tf)))
    if stage >= 6:
        steps.append(("sel", SelectFromModel(RandomForestClassifier(200, random_state=0), max_features=8, threshold=-np.inf)))
    clf = RandomForestClassifier(300, random_state=0, min_samples_leaf=2) if model == "rf" else LogisticRegression(max_iter=2000)
    if stage >= 7: return ImbPipeline(steps + [("smote", SMOTE(random_state=0)), ("clf", clf)])
    return Pipeline(steps + [("clf", clf)])

## 3. Stage by stage (repeated stratified 5-fold CV, mean ± std over 20 folds)

In [4]:
NAMES = ["0 raw numeric, median", "1 + KNN imputation", "2 + one-hot sex, embarked", "3 + scaling", "4 + IQR clipping",
         "5 + feature engineering", "6 + feature selection (8)", "7 + SMOTE"]
cols = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=0)
rows = []
for s, name in enumerate(NAMES):
    r = {m: cross_validate(make(s, m), T[cols], y, cv=cv, scoring=["accuracy", "f1"], n_jobs=-1) for m in ("rf", "lr")}
    rows.append({"stage": name,
                 "RF accuracy": f"{r['rf']['test_accuracy'].mean():.3f} +- {r['rf']['test_accuracy'].std():.3f}",
                 "LR accuracy": f"{r['lr']['test_accuracy'].mean():.3f} +- {r['lr']['test_accuracy'].std():.3f}",
                 "RF F1 (survived)": f"{r['rf']['test_f1'].mean():.3f} +- {r['rf']['test_f1'].std():.3f}"})
pd.DataFrame(rows).set_index("stage")

,RF accuracy,LR accuracy,RF F1 (survived)
stage,,,
"0 raw numeric, median",0.717 +- 0.036,0.700 +- 0.025,0.606 +- 0.058
1 + KNN imputation,0.721 +- 0.039,0.698 +- 0.023,0.614 +- 0.057
"2 + one-hot sex, embarked",0.829 +- 0.026,0.803 +- 0.028,0.764 +- 0.036
3 + scaling,0.829 +- 0.025,0.804 +- 0.028,0.765 +- 0.036
4 + IQR clipping,0.824 +- 0.026,0.797 +- 0.030,0.758 +- 0.035
5 + feature engineering,0.824 +- 0.026,0.802 +- 0.028,0.761 +- 0.036
6 + feature selection (8),0.832 +- 0.030,0.795 +- 0.027,0.771 +- 0.040
7 + SMOTE,0.825 +- 0.030,0.780 +- 0.031,0.774 +- 0.037


**Reading the table.** The only large jump is stage 2: adding `sex` (the strongest signal) and the port. Median against KNN imputation, scaling (irrelevant for a forest), clipping, the engineered features and selection all change the accuracy by less than one standard deviation of the fold-to-fold scatter (about 0.03). SMOTE does not raise accuracy; it can raise the F1 of the minority class a little. The first version of this notebook reported steady gains at every stage; most of those differences were noise on a single 5-fold split.

## 4. How much does leakage matter here? (and when it matters a lot)

In [5]:
# (a) Titanic: impute + select on ALL rows, then cross-validate (what the first notebook did)
d = engineer(T[cols]); d["sex"] = (d["sex"] == "male").astype(int)
d = pd.concat([d.drop(columns="embarked"), pd.get_dummies(T["embarked"], prefix="emb").astype(int)], axis=1)
X_all = pd.DataFrame(KNNImputer(n_neighbors=5).fit_transform(d), columns=d.columns)           # fitted on all rows
top = pd.Series(RandomForestClassifier(200, random_state=0).fit(X_all, y).feature_importances_, index=X_all.columns).nlargest(8).index
cv5 = StratifiedKFold(5, shuffle=True, random_state=0)
print("Titanic, selection on all data :", round(cross_val_score(RandomForestClassifier(300, random_state=0, min_samples_leaf=2), X_all[top], y, cv=cv5).mean(), 3))
print("Titanic, selection in pipeline :", round(cross_val_score(make(6), T[cols], y, cv=cv5).mean(), 3))

# (b) pure noise: 100 rows, 1000 random features, random labels. The true accuracy is 0.5.
rng = np.random.default_rng(0); Xn = rng.normal(size=(100, 1000)); yn = rng.integers(0, 2, 100)
leak = cross_val_score(LogisticRegression(max_iter=1000), SelectKBest(f_classif, k=20).fit_transform(Xn, yn), yn, cv=cv5).mean()
safe = cross_val_score(Pipeline([("sel", SelectKBest(f_classif, k=20)), ("lr", LogisticRegression(max_iter=1000))]), Xn, yn, cv=cv5).mean()
print(f"noise labels, selection outside CV: {leak:.3f}   inside the pipeline: {safe:.3f}")

Titanic, selection on all data : 0.836


Titanic, selection in pipeline : 0.827
noise labels, selection outside CV: 0.850   inside the pipeline: 0.570


On Titanic the leak is small because there are only a few strong features; with many weak features and few rows (a typical situation in medicine or signal processing) the same mistake turns noise into 80 to 85 % accuracy (true value 50 %). The rule is therefore unconditional: **everything that learns from data goes inside the pipeline.**

## 5. Exercises
- Add a `LogisticRegression` with `class_weight="balanced"` and compare its F1 with the SMOTE pipeline.
- Replace `SelectFromModel` by `SelectKBest(f_classif, k=8)` and by `Lasso`-based selection; compare, with paired differences over the folds.
- The `name` column of the original Kaggle file contains real titles (Mr, Mrs, Miss, Master). Extract them with a regular expression and test whether they help beyond `sex` and `age`.